In [1]:
import os
# Get the name of the current conda environment
env_name = os.getenv("CONDA_DEFAULT_ENV")
print(f"The current Conda environment is: {env_name}")

The current Conda environment is: tensorflow_env


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

In [3]:
epochs=1000
batch_size= 30
os.chdir('F:\\geodata\\river_runoff_obs')

In [4]:
input_file_name_list = ['1_hsg_imputMF','2_dsk_imputMF','3_xhl_imputMF','4_slglk_imputMF','5_kq_imputMF','6_wlwt_imputMF','7_tgzlk_imputMF']
input_file_name_list = [ '2_dsk_imputMF','3_xhl_imputMF','4_slglk_imputMF','5_kq_imputMF','6_wlwt_imputMF','7_tgzlk_imputMF']

In [5]:
for input_file_name in input_file_name_list:
    abbre = file_name.split('_')[1]
    for scenario in ['ssp126','ssp245','ssp370','ssp585']:
        print(scenario)
        
        # Dataset loading
        name = f'{input_file_name}_BCC-CSM2-MR_{scenario}_r1i1p1f1_daily'
        csv_path = f"{name}.csv"
        usecols = ['time', 'pre', 'tm',  'dis','ep']
        df_full = pd.read_csv(csv_path, usecols=usecols)
        df = df_full.dropna()
        df_future = df_full[df_full.dis.isnull()]
        file_name = f'{name}_{epochs}_{batch_size}'
        # Prepare features (X) and targets (y)
        X = df[['pre', 'tm']].values  # Inputs: precipitation, temperature, mass balance
        y = df[['dis', 'ep']].values         # Outputs: runoff (dis) and evaporation (ep)
        
        # Split the data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
        
        # Standardize the data
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        
        X_train = scaler_X.fit_transform(X_train)
        X_test = scaler_X.transform(X_test)
        y_train = scaler_y.fit_transform(y_train)
        y_test = scaler_y.transform(y_test)
        
        # Define the model
        model = Sequential([
            Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
            Dense(64, activation='relu'),
            Dense(32, activation='relu'),
            Dense(2)  # Output layer with two units for runoff and evaporation
        ])
        
        # Compile the model
        model.compile(optimizer='adam', loss='mse')
        
        # Train the model
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2)
        
        # Make predictions
        predictions = model.predict(X_test)
        predictions_rescaled = scaler_y.inverse_transform(predictions)
        
        # Convert predictions to DataFrame for runoff and evaporation
        pred_df = pd.DataFrame(predictions_rescaled, columns=['predicted_runoff', 'predicted_ep'])
        print(pred_df.head())
        from sklearn.metrics import mean_squared_error
        # Calculate predictions and rescale
        predictions = model.predict(X_test)
        predictions_rescaled = scaler_y.inverse_transform(predictions)
        
        # Separate the predicted and actual values for runoff and evaporation
        y_test_rescaled = scaler_y.inverse_transform(y_test)
        runoff_observed = y_test_rescaled[:, 0]
        evaporation_observed = y_test_rescaled[:, 1]
        runoff_predicted = predictions_rescaled[:, 0]
        evaporation_predicted = predictions_rescaled[:, 1]
        
        
        # Assuming future_df is loaded and has the same columns as df
        # Extract features from future_df
        X_full = df_full[['pre', 'tm']].values  # Only the input features
        # Standardize features based on the training data scaler
        X_full_scaled = scaler_X.transform(X_full)
        # Predict future runoff and evaporation
        full_predictions_scaled = model.predict(X_full_scaled)
        
        # Rescale predictions to original scale
        full_predictions = scaler_y.inverse_transform(full_predictions_scaled)
        
        # Convert predictions to DataFrame for readability
        df_full[['Projected_Runoff', 'Projected_Evaporation']] = full_predictions
        print(df_full[['Projected_Runoff', 'Projected_Evaporation']].head())
        
        # Convert the 'time' column to datetime format if needed
        df.loc[:,'time'] = pd.to_datetime(df['time'])
        df_full.loc[:,'time'] = pd.to_datetime(df_full['time'])
        # Ensure 'time' is set as the index for both DataFrames if not already
        df.set_index('time', inplace=True)
        df_full.set_index('time', inplace=True)
        historic_pred_df = df_full.dropna()
        historic_pred_df = historic_pred_df.tail(int(0.3 * len(historic_pred_df)))
        # Define the NSE function
        def nse(y_true, y_pred):
            return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
        # Calculate NSE for runoff and evaporation
        nse_runoff = nse(historic_pred_df.dis, historic_pred_df.Projected_Runoff)
        nse_evaporation = nse(historic_pred_df.ep, historic_pred_df.Projected_Evaporation)
        print('nse_runoff:',nse_runoff,'nse_ep:',nse_evaporation)
        
        # Plot the line chart for Projected_Runoff
        plt.plot(historic_pred_df.index, historic_pred_df['Projected_Runoff'], label='Projected Runoff', color='red')
        
        # Plot the scatter plot for dis
        plt.scatter(historic_pred_df.index, historic_pred_df['dis'], label='Observed Runoff', color='b')
        
        # Add labels, title, and legend
        plt.xlabel('Index')
        plt.ylabel(r'Runoff ($\mathrm{m^3 \cdot s^{-1}}$)')  # Use LaTeX for units
        plt.title('Projected Runoff vs Observed Runoff')
        plt.legend()
        
        # Show the plot
        plt.show()
        
        # Plot with a larger figure size
        ax = df_full[[ 'Projected_Runoff','dis']].plot(figsize=(12, 6))
        # Optional: Rotate x-tick labels for better readability
        plt.xticks(rotation=45)
        plt.title(f"Historical and projected runoff of {file_name}")
        # Show the plot
        plt.show()
        # Plot with a larger figure size
        ax = df_full[['ep', 'Projected_Evaporation']].plot(figsize=(12, 6))
        # Optional: Rotate x-tick labels for better readability
        plt.xticks(rotation=45)
        plt.title(f"Historical and projected evaporation of {file_name}")
        # Show the plot
        plt.show()
        import matplotlib.pyplot as plt
        
        # Define start and end dates as datetime objects
        start_date = pd.to_datetime("2000-01-01")
        end_date = pd.to_datetime("2100-12-31")
        
        # Generate a range of ticks every 20 years
        x_ticks = pd.date_range(start=start_date, end=end_date, freq='10Y')
        
        # Set the size of the figure
        fig, axes = plt.subplots(5, 1, figsize=(16, 9), sharex=True,facecolor='w')
        title_list = ['Monthly Precipitation','Monthly Temperature','Monthly Glacier Runoff','Monthly Runoff','Monthly Evaporation']
        
        # Define y-axis limits for each plot
        y_lims = [(0, 70), (-20, 25), (0, 3.3* 1e9), (0, 1200), (0, 800)]
        
        # Loop through the columns and set y-axis limits
        for i, col in enumerate(['pre', 'tm', ['dis', 'Projected_Runoff'], ['ep', 'Projected_Evaporation']]):
            df_full[col].plot(ax=axes[i])
            
            # Set y-axis limits
            axes[i].set_ylim(y_lims[i])
            
            # Optional: Add y-axis label, legend, title, grid, etc.
            # axes[i].set_ylabel(col)  # Set y-axis label to the column name
            axes[i].legend(loc='upper right')  # Optional: add legend
            axes[i].set_title(title_list[i])
            
            # Set x-axis limits
            axes[i].set_xlim([start_date, end_date])
            axes[i].set_xticks(x_ticks)
            
            # Set x-axis labels as years (2000, 2020, ..., 2100)
            axes[i].set_xticklabels([str(date.year) for date in x_ticks], rotation=0)
            
            axes[i].grid(True)  # Optional: add grid for clarity
        
        
        
        
        # Set x-axis label for the entire figure
        axes[-1].set_xlabel('Date')  # or adjust label based on your x-axis
        
        # Adjust layout to prevent overlap
        plt.tight_layout()
        plt.savefig(f'{file_name}.svg')
        plt.show()
        
        df_full.to_csv(f"{file_name}_project.csv")
        annual_df = df_full.resample('Y').agg({
            'pre': 'sum',  # If there are any NaNs in the group, the sum will be NaN
            'tm': 'mean',  # The mean will also return NaN if there are NaNs in the group
            'dis': 'sum',
            'ep': 'sum',
            'Projected_Runoff': 'sum',
            'Projected_Evaporation': 'sum'
        }, skipna=False)  # Ensures NaNs are preserved in aggregation
        
        # Reset index if needed
        annual_df.reset_index(inplace=True)
        annual_df[['ep', 'dis', 'Projected_Runoff',  'Projected_Evaporation']]=annual_df[['ep', 'dis', 'Projected_Runoff',  'Projected_Evaporation']].replace(0, np.nan,)
        
        # Display the result
        print(annual_df.head())
        
        annual_df[['dis', 'Projected_Runoff']].plot()
        # plt.ylim(bottom=20000)

ssp126


ValueError: Usecols do not match columns, columns expected but not found: ['ep']